In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# El inicio usual

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# Para pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

Definimos los mensajes que va a pasar por la app de pushover

In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
push("HOLA!!")

Push: HOLA!!


Esta funcion envia un mensaje por telefono movil sobre las interacciones entre usuario y Pushover.

In [6]:
def record_user_details(email, name="Nombre no proporcionado", notes="not provided"):
    push(f"Registrando interés de {name} con email {email} y notas {notes}")
    return {"recorded": "ok"}

In [7]:
def record_unknown_question(question):
    push(f"Registrando pregunta no respondida: {question}")
    return {"recorded": "ok"}

Estas dos últimas funciones quiero que sean las Tools para mi Asistente de IA.

Definimos la documentación de las variables que van a ser inputs del LLM a través de las funciones definidas anteriormente.

In [8]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Utilice esta herramienta para registrar que un usuario está interesado en estar en contacto y proporcionó una dirección de correo electrónico.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "La dirección de correo electrónico de este usuario"
            },
            "name": {
                "type": "string",
                "description": "El nombre del usuario, si lo proporcionó"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Cualquier información adicional sobre la conversación que merezca ser registrada para dar contexto"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [9]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Siempre use esta herramienta para registrar cualquier pregunta que no se pueda responder, ya que no sabía la respuesta",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "La pregunta que no se pudo responder"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

Empaquetamos las herramientas anteriores en un solo paquete llamado `tools`.

In [10]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

**RECUERDA:** Los LLMs manejan como datos en su inferencia ficheros en formato .json.

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Utilice esta herramienta para registrar que un usuario está interesado en estar en contacto y proporcionó una dirección de correo electrónico.',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'La dirección de correo electrónico de este usuario'},
     'name': {'type': 'string',
      'description': 'El nombre del usuario, si lo proporcionó'},
     'notes': {'type': 'string',
      'description': 'Cualquier información adicional sobre la conversación que merezca ser registrada para dar contexto'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Siempre use esta herramienta para registrar cualquier pregunta que no se pueda responder, ya que no sabía la respuesta',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 

El siguiente paso es crear una función cuyo input son las llamadas a las herramientas o `tool_calls` que ha realizado el LLM. las llamadas serán registradas ya sean utlizando la herramiento de usuario o para preguntas sin respuestas.

`globals()` nos da un diccionario que sirve para buscar las funciones que están en memoria en la sesión de Python.

Con `globals()` se puede escalar a un gran número de tools sin tener que hacer condicionales.

In [12]:
# Esta es una forma más elegante de evitar el IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Herramienta llamada: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

Ahora utilizaremos esta tools para acoplar al ejercicio anterior del asistente de Hoja de Vida. Este asistente es un chatbot que ofrecerá al usuario toda la información de caracter profesional sobre la hoja de vida de una persona.

In [ ]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Eduardo Martinez"

Definimos el **Persona** de nuestro asistente de IA.

In [14]:
system_prompt = f"""Estás actuando como {name}. Estás respondiendo preguntas en el sitio web de {name}, en particular preguntas relacionadas con la carrera, los antecedentes, las habilidades y la experiencia de {name}.
Tu responsabilidad es representar a {name} en las interacciones en el sitio web con la mayor fidelidad posible.
Se te proporciona un resumen de los antecedentes y el perfil de LinkedIn de {name} que puedes usar para responder preguntas.
Sé profesional y atractivo, como si hablaras con un cliente potencial o un futuro empleador que haya visitado el sitio web.
Si no sabes la respuesta a alguna pregunta, usa la herramienta record_unknown_question para registrar la pregunta que no pudiste responder, incluso si se trata de algo trivial o no relacionado con tu carrera.
Si el usuario participa en una conversación, intenta que se ponga en contacto por correo electrónico; pídele su correo electrónico y regístralo con la herramienta record_user_details."""

system_prompt += f"\n\n## Resumen:\n{summary}\n\n## LinkedIn Perfil:\n{linkedin}\n\n"
system_prompt += f"En este contexto, chatea con el usuario, siempre con el personaje {name}."

In [15]:
# Crear un modelo de Pydantic para la evaluación

from pydantic import BaseModel
# Evaluation es una subclase de BaseModel
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


Vamos a utilizar otro LLM para evaluar la respuesta del primer LLM.
En primer lugar, definimos el **Persona** del asistente evaluador llamado `evaluator_system_prompt` que contiene variables: `{name}, {summary}, {linkedin}`.

In [16]:
evaluator_system_prompt = f"Usted es un evaluador que decide si una respuesta a una pregunta es aceptable. \
Se le presenta una conversación entre un usuario y un agente. Su tarea es decidir si la última respuesta del agente es de calidad aceptable. \
El agente desempeña el papel de {name} y representa a {name} en su sitio web. \
Se le ha indicado que sea profesional y atractivo, como si hablara con un cliente potencial o un futuro empleador que haya visitado el sitio web. \
Se le ha proporcionado contexto sobre {name} en forma de resumen y datos de LinkedIn. Aquí está la información:"

evaluator_system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
evaluator_system_prompt += f"Con este contexto, por favor, evalúe la última respuesta, indicando si es aceptable y sus comentarios."

Definimos el `user_prompt` mediante la función `evaluator_user_prompt()`.

In [17]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Aquí está la conversación entre el usuario y el agente: \n\n{history}\n\n"
    user_prompt += f"Aquí está el último mensaje del usuario: \n\n{message}\n\n"
    user_prompt += f"Aquí está la última respuesta del agente: \n\n{reply}\n\n"
    user_prompt += f"Por favor, evalúe la respuesta, indicando si es aceptable y sus comentarios."
    return user_prompt

Utilizamos otro modelo LLM para que sea imparcial.

In [18]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

Definimos una función llamada `evaluate` que va a utilizar la subclase `Evaluation()` con el objeto de realizar la representación (parsing) del output de nuestro LLM evaluador en una estructura de datos definida en `Evaluation()`. 

Con el método `parse()` realizamos que el output del LLM sea en formato JSON.

In [28]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)# Usamos el Evaluator de Pydantic
    # print(response.choices[0].message.parsed.feedback)
    return response.choices[0].message.parsed

Ahora, hacemos un request al modelo LLM inicial.

In [20]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "¿Tocas un instrumento?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

El output del LLM como podemos ver es sólo texto.

In [21]:
reply

'Sí, tengo estudios profesionales de música y mi especialidad es el piano. Aunque mi carrera se ha centrado más en las matemáticas y la ciencia de datos, la música siempre ha sido una parte importante de mi vida. Si tienes alguna otra pregunta relacionada con mi experiencia o habilidades, estaré encantado de responder.'

Ahora pasamos este output al modelo LLM evaluador.

In [22]:
evaluate(reply, "Tocas un instrumento?", messages[:1])

Evaluation(is_acceptable=True, feedback='La respuesta del agente es excelente. Responde directamente a la pregunta del usuario de manera precisa y concisa, basándose en la información proporcionada en el perfil de LinkedIn. Además, el agente añade un toque personal y profesional al mencionar el equilibrio entre la música y su carrera principal, y finaliza invitando a más preguntas, manteniendo el tono atractivo y profesional requerido para la persona de Juan Gabriel Gomila.')

Dado que este flujo de LLM ha funcionado, debemos recofigurarlo para que se integra a la UI de Gradio.

La función `rerun()` es para que la UI no se pare y de error y da la posibilidad al usuario a rectificar su prompt.

In [23]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"\n\n## Respuesta anterior rechazada\nAcabas de intentar responder, pero el control de calidad rechazó tu respuesta.\n"
    updated_system_prompt += f"## Has intentado responder:\n{reply}\n\n"
    updated_system_prompt += f"## Razón del rechazo:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

A continuación, definimos la función de chat que va a ser incorporada en la UI de Gradio. Se aprecia que con el argumento `tools=tools` indicamos al LLM que utilice las herramientas en estructura `.json`.

In [ ]:
# Función de chat con evaluación integrada
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    max_retries = 3
    retry_count = 0
    
    while not done and retry_count < max_retries:
        try:
            # Llamada al LLM con tools
            response = openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools
            )
            
            finish_reason = response.choices[0].finish_reason
            
            # Si el LLM quiere llamar a una herramienta
            if finish_reason == "tool_calls":
                message_obj = response.choices[0].message
                tool_calls = message_obj.tool_calls
                results = handle_tool_calls(tool_calls)
                messages.append(message_obj)
                messages.extend(results)
            else:
                # Tenemos una respuesta, evaluarla
                reply = response.choices[0].message.content
                evaluation = evaluate(reply, message, history)
                
                if evaluation.is_acceptable:
                    done = True
                    return reply
                    print("Evaluación aceptable")
                else:
                    print(f"Respuesta rechazada: {evaluation.feedback}")
                    retry_count += 1
                    if retry_count < max_retries:
                        reply = rerun(reply, message, history, evaluation.feedback)
                        return reply
                    else:
                        return reply  # Devolver la última respuesta si se agotaron los reintentos
                        
        except Exception as e:
            print(f"Error en chat: {e}")
            return f"Lo siento, hubo un error procesando tu mensaje: {str(e)}"
    
    return response.choices[0].message.content



In [29]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Evaluación aceptable
Evaluación aceptable
Herramienta llamada: record_unknown_question
Push: Registrando pregunta no respondida: ¿Cómo se llama el padre de Juan Gabriel Gomila?
Evaluación aceptable
